# Stokes Problem Analysys

In [1]:
import os
import gdown

import torch

from podcnf.NFmodel import NormalizingFlow

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

## Load Model and Data

In [2]:
# # If you have data in the data folder you can upload it, same for the model in the results folder
# input_file = "../data/stokes_data_6400.pt" 
# loaded_data = torch.load(input_file, weights_only=True)

# # Model
# model_file = "../results/stokes/NF_Stokes.pth"
# loaded_model = torch.load(model_file, map_location = device)
# loaded_params = loaded_model['hyperparameters']

# # mu and c for the initialization of the model
# reduced_input_file = "../data/stokes_data_reduced_6400.pt" 

# reduced_dataset = torch.load(reduced_input_file, weights_only=True)
# mu = reduced_dataset['mu'].numpy()
# c = reduced_dataset['c'].numpy()

In [3]:
os.makedirs('../data', exist_ok=True)
target_folder = os.path.join("..", "data") 
reduced_input_file = os.path.join(target_folder, "stokes_data_reduced_6400.pt")
gdown.download(id="19304ojlsmuL7hntN-m8KeR_CrAZD7wHb", quiet=True, output=reduced_input_file)
reduced_dataset = torch.load(reduced_input_file, weights_only=True)
mu = reduced_dataset['mu'].numpy()
c = reduced_dataset['c'].numpy()

In [ ]:
os.makedirs('../data', exist_ok=True)
target_folder = os.path.join("..", "data")
filename = os.path.join(target_folder, "stokes_data_6400.pt")
gdown.download(id="1_E-gNMU9aMmWXHqm63JspI9i4kWy-wd1", quiet=True, output=filename)
loaded_data = torch.load(filename, weights_only=True)

In [6]:
os.makedirs('../results', exist_ok=True)
target_folder = os.path.join("..", "results")
MODEL_NAME = os.path.join(target_folder, "NF_Stokes.pth")
gdown.download(id = "1o3TPXo87eKOHRo82LjsOxeVTWZ6Od3mQ", quiet=True, output = MODEL_NAME)
loaded_model = torch.load(MODEL_NAME, map_location = device)
loaded_params = loaded_model['hyperparameters']

In [7]:
# Extract the tensors
u = loaded_data['u']
eps = loaded_data['eps'].squeeze(1)
theta = loaded_data['theta'].squeeze(1)
mu = loaded_data['mu']

# Verify the shapes
print(f"u shape: {u.shape}")
print(f"mu shape: {mu.shape}")
print(f"eps shape: {eps.shape}")
print(f"theta shape: {theta.shape}")

u shape: torch.Size([6400, 2763])
mu shape: torch.Size([6400, 3])
eps shape: torch.Size([6400, 1])
theta shape: torch.Size([6400, 1])


In [8]:
dim_x = mu.shape[1]
dim_y = c.shape[1]
num_flows_loaded = loaded_params['num_flows']
hidden_size_loaded = loaded_params['hidden_size']
hidden_depth_loaded = loaded_params['hidden_depth']

print(f"NF parameters flows: {num_flows_loaded}, size: {hidden_size_loaded}, depth: {hidden_depth_loaded}")

NF_linear = NormalizingFlow(
    dim_x,
    dim_y,
    num_flows=num_flows_loaded,
    hidden_size=hidden_size_loaded,
    hidden_depth=hidden_depth_loaded,
    device=device
).to(device)

NF_linear.load_state_dict(loaded_model['model_state_dict'])

NF parameters flows: 24, size: 128, depth: 2


<All keys matched successfully>

## Performance on a fixed $\mu$

### Monte Carlo for $\mathbb{E}[u|\mu]$